In [1]:
!pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 825.1 kB/s eta 0:00:02
   -------------- ------------------------- 0.5/1.5 MB 825.1 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   --------------------- ------------------ 0.8/1.5 MB 620.2 kB/s eta 0:00:02
   ----------------------------

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
data=pd.read_csv('TRAIN.csv')
data.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [4]:
X=data.drop('Class',axis=1)
y=data['Class']

In [1]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.25,random_state=42)

NameError: name 'X' is not defined

In [6]:
from lightgbm import LGBMClassifier

In [7]:
lgb_model=LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    class_weight='balanced',
    random_state=42
)

In [8]:
lgb_model.fit(X_train,y_train)

[LightGBM] [Info] Number of positive: 4279, number of negative: 6665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004599 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 10944, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=500,
               random_state=42)

In [9]:
y_prob=lgb_model.predict_proba(X_test)[:,1]
y_pred=lgb_model.predict(X_test)

In [18]:
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score,confusion_matrix,f1_score

In [19]:
thresholds=np.arange(0.1,0.09,0.01)
best_f1 = 0
best_t = 0

for t in thresholds:
    preds = (y_prob > t).astype(int)
    f1 = f1_score(y_test, preds)
    
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(best_t, best_f1)

0 0


In [11]:
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))

0.9753898635477583
[[19540   260]
 [  548 12484]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     19800
           1       0.98      0.96      0.97     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.97      0.97     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.9966731929485151


In [12]:
param_grid = {
    "num_leaves":[31,63,127],
    "learning_rate":[0.01,0.03,0.05],
    "n_estimators":[400,600,800],
    "subsample":[0.7,0.8,0.9],
    "colsample_bytree":[0.7,0.8,0.9]
}

In [13]:
from sklearn.model_selection import RandomizedSearchCV

random_search=RandomizedSearchCV(
   lgb_model,
   param_grid,
   n_iter=25,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1 
)

In [14]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[LightGBM] [Info] Number of positive: 4279, number of negative: 6665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005958 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 10944, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


RandomizedSearchCV(cv=5,
                   estimator=LGBMClassifier(class_weight='balanced',
                                            learning_rate=0.05,
                                            n_estimators=500, random_state=42),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9],
                                        'learning_rate': [0.01, 0.03, 0.05],
                                        'n_estimators': [400, 600, 800],
                                        'num_leaves': [31, 63, 127],
                                        'subsample': [0.7, 0.8, 0.9]},
                   scoring='f1', verbose=1)

In [15]:
y_prob_hp=random_search.predict_proba(X_test)[:,1]
y_pred_hp=random_search.predict(X_test)

In [16]:
print(accuracy_score(y_test,y_pred_hp))
print(confusion_matrix(y_test,y_pred_hp))
print(classification_report(y_test,y_pred_hp))
print("AUC:", roc_auc_score(y_test, y_prob_hp))

0.9807809454191033
[[19619   181]
 [  450 12582]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     19800
           1       0.99      0.97      0.98     13032

    accuracy                           0.98     32832
   macro avg       0.98      0.98      0.98     32832
weighted avg       0.98      0.98      0.98     32832

AUC: 0.9980306711994098


In [17]:
from sklearn.model_selection import cross_val_score

score=cross_val_score(lgb_model,X_train,y_train,cv=5,scoring='f1')

print(score)
print(score.mean())

[LightGBM] [Info] Number of positive: 3423, number of negative: 5332
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005569 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 8755, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 3423, number of negative: 5332
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003309 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 8755, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Nu